In [ ]:
%pip install google-cloud-storage
%pip install numpy
%pip install torch
%pip install transformers
%load_ext autoreload
%autoreload 2
%pip install accelerate

In [ ]:
import io
import numpy as np
from google.cloud import storage

def load_npz_from_gcs(bucket_name, blob_name):
    # 1. Initialize the GCS client (uses your GOOGLE_APPLICATION_CREDENTIALS)
    client = storage.Client(project="gen-lang-client-0105254213")
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    # 2. Download the contents as bytes
    content = blob.download_as_bytes()

    # 3. Load into NumPy using a BytesIO buffer
    # Use 'with' to ensure the NpzFile object is closed correctly
    with np.load(io.BytesIO(content)) as data:
        # Access your features (e.g., data['features'])
        return {key: data[key] for key in data.files}

# Example Usage
# For path: gs://meld/hubert_features/dia0_utt0.npz
features = load_npz_from_gcs("meld", "hubert_features/meld_train_hubert.npz")
print(f"Loaded keys: {features.keys()}")
print(f"Features Shape: {features['features'].shape}")
print(f"Labels Shape: {features['labels'].shape}")
print(f"Sample Emotion Names: {features['emotion_names'][:7]}")

"""
Loaded keys: dict_keys(['features', 'labels', 'ids', 'emotion_names'])
Features Shape: (9988, 1024)
Labels Shape: (9988,)
Sample Emotion Names: ['anger' 'disgust' 'fear' 'joy' 'neutral' 'sadness' 'surprise']

"""

In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

# login(tok")
model_name = "Qwen/Qwen3.5-2B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant specialized in MELD character analysis."},
    {"role": "user", "content": "Describe Ross Geller's emotional baseline in one sentence."}
]
model.resize_token_embeddings(len(tokenizer))
model = model.to("cuda")
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# 4. Generate
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=True,      # Mandatory for Qwen 3.5/Thinking models
    temperature=0.6,     # Qwen's official recommendation for thinking
    top_p=0.95,          # Helps keep the output coherent
    pad_token_id=tokenizer.pad_token_id
)
new_tokens = generated_ids[0][model_inputs.input_ids.shape[-1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)
print(response)

In [ ]:
def generate_qwen_bio(speaker_name, scene_transcript):
    prompt = f"""
    You are an expert in character analysis for the TV show Friends. 
    Based on the following scene transcript, describe the speaker '{speaker_name}' 
    focusing on their typical emotional baseline, vocal habits, and personality traits.
    
    Context: {scene_transcript}
    
    Character Biography for {speaker_name}:
    """
    # Call Qwen 3.5 here...
    return qwen_response